# 🎬 CineData Analytics — Agente Text-to-SQL

Agente que transforma perguntas em linguagem natural em consultas SQL de leitura sobre a **camada Gold** da CineData (SQLite) e responde em português.

**Como funciona**
1. A pergunta é enviada ao LLM (OpenRouter, modelos `:free`) junto com o schema e as regras de negócio.
2. O LLM chama a ferramenta `execute_sql` com uma consulta SQLite.
3. A consulta passa pelos guardrails (conexão read-only + autorizador que só permite leitura) e é executada.
4. O LLM lê o resultado (corrigindo a query se houve erro) e redige a resposta.

> ⚠️ A conta gratuita do OpenRouter permite **50 requisições/dia** e cada pergunta usa ~2. As respostas ficam em cache (`.cache/`), então reexecutar o notebook não gasta cota.

## 1. Configuração

In [ ]:
import logging
import pandas as pd
from IPython.display import Markdown, display

from cinedata.agent import CineDataAgent

logging.basicConfig(level=logging.WARNING)
agent = CineDataAgent()
print("Banco:", agent.settings.db_path)
print("Modelos (ordem de fallback):", agent.settings.models)

In [ ]:
def perguntar(pergunta: str, nova_conversa: bool = True):
    """Faz uma pergunta ao agente e exibe o SQL, a resposta e a tabela retornada."""
    if nova_conversa:
        agent.reset()
    r = agent.ask(pergunta)
    display(Markdown(f"### ❓ {pergunta}"))
    for i, step in enumerate(r.steps, 1):
        status = f"❌ {step.error}" if step.error else "✅"
        display(Markdown(f"**SQL {i}** {status}\n```sql\n{step.query}\n```"))
    display(Markdown(r.answer))
    display(Markdown(f"<sub>fonte: {'cache' if r.from_cache else r.model}</sub>"))
    if r.last_result:
        return pd.DataFrame(r.last_result.rows, columns=r.last_result.columns)

## 2. Perguntas de exemplo da atividade

### 💰 Análise de Bilheteria e Finanças

In [ ]:
perguntar("Quais são os 10 filmes com maior receita em R$?")

In [ ]:
perguntar("Qual é o lucro médio por gênero, considerando apenas filmes com receita informada?")

In [ ]:
perguntar("Quais filmes têm a maior margem de lucro, entre os que possuem receita e orçamento informados?")

### ⭐ Análise de Popularidade e Engajamento

In [ ]:
perguntar("Quais são os 5 filmes mais populares?")

In [ ]:
perguntar("Quais filmes têm a maior divergência entre a nota TMDB e a nota IMDb?")

In [ ]:
perguntar("Qual é a nota média IMDb por ano de lançamento?")

### 🎭 Análise de Elenco e Equipe

In [ ]:
perguntar("Qual ator participou de mais filmes lançados nos últimos 5 anos?")

In [ ]:
perguntar("Quais diretores têm a maior nota média, considerando no mínimo 5 filmes?")

In [ ]:
perguntar("Qual dupla ator-diretor mais trabalhou junta?")

### 🏢 Análise de Gêneros e Produtoras

In [ ]:
perguntar("Qual é a quantidade de filmes por gênero?")

In [ ]:
perguntar("Qual produtora teve o maior lucro total?")

In [ ]:
perguntar("Qual gênero tem a maior margem de lucro média?")

### 📝 Análise de Avaliações dos Usuários

In [ ]:
perguntar("Quais são os filmes mais avaliados pelos usuários?")

In [ ]:
perguntar("Em quais filmes a nota média dos usuários mais diverge da nota IMDb?")

## 3. Memória de conversa
Com `nova_conversa=False` o agente lembra do contexto anterior e entende perguntas de acompanhamento.

In [ ]:
perguntar("Quais são os 3 gêneros com mais filmes?")

In [ ]:
perguntar("E qual a nota IMDb média de cada um deles?", nova_conversa=False)

## 4. Guardrails
Pedidos de escrita são recusados pelo LLM e, mesmo que ele tentasse, o banco bloquearia. Abaixo, o teste direto na camada de banco (sem gastar cota):

In [ ]:
from cinedata.database import QueryError

for sql in ["DELETE FROM dim_movies", "DROP TABLE dim_genres", "SELECT 1; DROP TABLE dim_genres"]:
    try:
        agent.db.run_query(sql)
    except QueryError as e:
        print(f"Bloqueado: {sql!r} -> {e}")

## 5. Pergunte você mesmo

In [ ]:
perguntar("Quais os 5 filmes de terror com maior nota IMDb, com pelo menos 1000 votos?")